In [ ]:
# Databricks Notebook: Delta Lake Maintenance — OPTIMIZE, ZORDER, VACUUM, DESCRIBE HISTORY
# 스케줄링: Databricks Workflows Job으로 주 1회(일요일 02:00 UTC) 실행 권장
# 실행 순서: Gold 파이프라인(03a-03d) 완료 후 이 노트북을 실행
#
# OPTIMIZE: 소파일들을 ~1GB Parquet 파일로 병합 (Delta default target size)
# ZORDER:   지정 컬럼 기준으로 행을 클러스터링 → 파일 수준 min/max 통계 개선
#            → 대시보드 필터(symbol, bucket_start) 쿼리 시 Data Skipping으로 파일 스캔 최소화
# VACUUM:   Delta 트랜잭션 로그에서 더 이상 참조하지 않는 Parquet 파일 삭제
#            → 보존 기간(hours) 내 버전은 Time Travel로 조회 가능

from delta.tables import DeltaTable

CATALOG = "demo_catalog"
SCHEMA  = "demo_schema"

# 테이블명: (zorder_cols, vacuum_retain_hours)
# zorder_cols=None: ZORDER 없이 OPTIMIZE만 수행
# vacuum_retain_hours: 최소 168(7일) 권장. Gold는 14일(336h) 유지 → Time Travel 창 확보
TABLES = {
    "bronze_charts":                           (None,                          168),
    "bronze_fear_greed":                        (None,                          168),
    "silver_charts":                           (["symbol", "open_time"],       168),
    "silver_fear_greed":                       (["dt"],                         168),
    "gold_prices_4h":                          (["symbol", "bucket_start"],     336),
    "gold_fear_greed":                         (["dt"],                         336),
    "gold_price_positions_4h":                 (["symbol", "bucket_start"],     336),
}

print(f"[MAINTENANCE] 대상 테이블 수: {len(TABLES)}")
print(f"[MAINTENANCE] Catalog: {CATALOG}.{SCHEMA}")

In [ ]:
# ===== OPTIMIZE + ZORDER =====
# OPTIMIZE: Delta가 소파일들을 ~1GB 단위로 병합. 쓰기 성능보다 읽기 성능 우선.
# ZORDER:   지정 컬럼의 값이 유사한 행을 같은 파일 내에 물리적으로 모음.
#           → Delta가 각 파일의 min/max 통계를 유지하므로,
#             WHERE symbol='BTCUSDT' AND bucket_start > '...' 쿼리 시
#             해당 범위를 포함하지 않는 파일은 스캔 자체를 건너뜀 (Data Skipping)
# 주의: OPTIMIZE는 실행 시간이 길 수 있으므로 오프피크 시간대 스케줄 권장

print("[OPTIMIZE] 시작...")
for table_name, (zorder_cols, _) in TABLES.items():
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    try:
        if zorder_cols:
            zorder_clause = ", ".join(zorder_cols)
            sql = f"OPTIMIZE {full_name} ZORDER BY ({zorder_clause})"
            print(f"  [OPTIMIZE+ZORDER] {full_name} → ZORDER BY ({zorder_clause})")
        else:
            sql = f"OPTIMIZE {full_name}"
            print(f"  [OPTIMIZE] {full_name}")
        spark.sql(sql)
    except Exception as e:
        print(f"  [WARN] {full_name} OPTIMIZE 실패 (테이블이 아직 없을 수 있음): {e}")

print("[OPTIMIZE] 모든 테이블 완료")

In [ ]:
# ===== VACUUM =====
# VACUUM은 Delta 트랜잭션 로그에서 더 이상 참조되지 않으면서
# 보존 기간(RETAIN N HOURS)을 초과한 Parquet 파일을 물리적으로 삭제
#
# 보존 기간 설계:
#   - Bronze/Silver: 7일(168h) — 재수집/재변환이 용이하므로 짧게 유지
#   - Gold: 14일(336h) — 대시보드 소스. 데이터 이상 탐지 후 롤백 창 확보
#   - DLQ: 30일(720h) — 감사 추적 목적으로 장기 보존
#
# 경고: RETAIN 기간을 7일(168h) 미만으로 설정하면 Structured Streaming
#       체크포인트와 충돌할 수 있음. 안전 검사를 비활성화하지 않는 이상 최소 168h 권장.

# 안전장치: 실제 삭제 전 DRY RUN으로 삭제 대상 파일 수를 먼저 확인.
# 예상보다 훨씬 많은 파일이 삭제 대상이면(예: 보존기간 설정 실수) 실제 VACUUM을 건너뛰고 조사할 수 있도록 함.
DRY_RUN_ONLY = False  # True로 두면 삭제 없이 대상 파일 목록/수만 확인

print("[VACUUM] DRY RUN 확인...")
dry_run_counts = {}
for table_name, (_, vacuum_hours) in TABLES.items():
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    try:
        dry_df = spark.sql(f"VACUUM {full_name} RETAIN {vacuum_hours} HOURS DRY RUN")
        n = dry_df.count()
        dry_run_counts[full_name] = n
        print(f"  [DRY RUN] {full_name}: 삭제 예정 파일 {n}개 (RETAIN {vacuum_hours}h)")
    except Exception as e:
        print(f"  [WARN] {full_name} DRY RUN 실패: {e}")

print("[VACUUM] 시작...")
for table_name, (_, vacuum_hours) in TABLES.items():
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    if DRY_RUN_ONLY:
        continue
    try:
        print(f"  [VACUUM] {full_name} RETAIN {vacuum_hours} HOURS")
        spark.sql(f"VACUUM {full_name} RETAIN {vacuum_hours} HOURS")
    except Exception as e:
        print(f"  [WARN] {full_name} VACUUM 실패: {e}")

print("[VACUUM] 모든 테이블 완료" if not DRY_RUN_ONLY else "[VACUUM] DRY_RUN_ONLY=True → 실제 삭제는 건너뜀")

In [ ]:
# ===== DESCRIBE HISTORY (감사 로그 & Time Travel 진입점) =====
# DESCRIBE HISTORY는 Delta 트랜잭션 로그의 모든 작업을 조회:
#   version, timestamp, operation (WRITE/MERGE/OPTIMIZE/VACUUM 등),
#   operationMetrics (numFilesAdded, numFilesRemoved, numRowsInserted 등)
#
# Time Travel 활용:
#   - VERSION AS OF <n>: 특정 버전으로 쿼리 또는 RESTORE
#   - TIMESTAMP AS OF '<ts>': 특정 시점의 데이터 재현
#   - RESTORE: 잘못된 Gold 쓰기 후 이전 상태로 롤백

KEY_TABLES = [
    "gold_prices_4h",
    "gold_fear_greed",
    "gold_price_positions_4h",
]

for table_name in KEY_TABLES:
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    print(f"\n=== HISTORY: {full_name} ===")
    try:
        spark.sql(f"DESCRIBE HISTORY {full_name}").show(10, truncate=False)
    except Exception as e:
        print(f"  [WARN] {full_name} HISTORY 조회 실패: {e}")

# ===== Time Travel 예시 (주석 해제 후 사용) =====
# --- 특정 버전으로 Gold 쿼리 (파이프라인 재실행 전후 비교) ---
# spark.sql(f"""
#   SELECT * FROM {CATALOG}.{SCHEMA}.gold_prices_4h VERSION AS OF 3
#   WHERE symbol = 'BTCUSDT'
#   ORDER BY bucket_start DESC LIMIT 20
# """).show(truncate=False)
#
# --- 특정 타임스탬프 기준 Gold 스냅샷 재현 (대시보드 상태 감사) ---
# spark.sql(f"""
#   SELECT * FROM {CATALOG}.{SCHEMA}.gold_prices_4h
#   TIMESTAMP AS OF '2025-12-01 00:00:00'
#   WHERE symbol = 'ETHUSDT'
#   ORDER BY bucket_start DESC LIMIT 10
# """).show(truncate=False)
#
# --- Gold 테이블 롤백 (잘못된 파이프라인 실행 후 복구) ---
# spark.sql(f"RESTORE TABLE {CATALOG}.{SCHEMA}.gold_prices_4h TO VERSION AS OF 5")
# print("[RESTORE] 롤백 완료")

In [ ]:
# ===== DESCRIBE DETAIL (테이블 상태 확인) =====
# OPTIMIZE 전후로 이 셀을 실행하여 numFiles 감소를 확인
# sizeInBytes: 총 데이터 크기
# numFiles: Parquet 파일 수 (OPTIMIZE 후 대폭 감소 예상)
# partitionColumns: 파티셔닝 컬럼 확인

for table_name in KEY_TABLES:
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    print(f"\n=== DETAIL: {full_name} ===")
    try:
        spark.sql(f"DESCRIBE DETAIL {full_name}").select(
            "name", "numFiles", "sizeInBytes", "partitionColumns"
        ).show(truncate=False)
    except Exception as e:
        print(f"  [WARN] {full_name} DETAIL 조회 실패: {e}")

print("\n[MAINTENANCE] 모든 작업 완료")
print("  OPTIMIZE + ZORDER: 소파일 병합 및 데이터 클러스터링 완료")
print("  VACUUM: 보존 기간 초과 파일 삭제 완료")
print("  다음 실행: 1주일 후 (Databricks Workflows 스케줄 참고)")